Calculate the following questions using femwell mode solver:

- What is the maximum width for a Silicon Nitride (n=2) of 400 nm thick, fully etched nitride
waveguide to be single mode for TE at 1550 nm?

In [ ]:
from collections import OrderedDict

import matplotlib.pyplot as plt
import numpy as np
from shapely.geometry import box
from shapely.ops import clip_by_rect
from skfem import Basis, ElementTriP0
from skfem.io.meshio import from_meshio
from tqdm import tqdm

from femwell.maxwell.waveguide import compute_modes
from femwell.mesh import mesh_from_OrderedDict

In [ ]:
wg_thickness = 0.4  # As given by the task
n_SiN = 2 # As given by the task
n_SiO2 = 1.444
n_air = 1

wavelength = 1.55
num_modes = 8
widths = np.linspace(0.5, 3.5, 100)

all_neffs = np.zeros((widths.shape[0], num_modes))
all_te_fracs = np.zeros((widths.shape[0], num_modes))
for i, width in enumerate(tqdm(widths)):
    core = box(0, 0, width, wg_thickness)
    polygons = OrderedDict(
        core=core,
        box=clip_by_rect(core.buffer(1.0, resolution=4), -np.inf, -np.inf, np.inf, 0),
        clad=clip_by_rect(core.buffer(1.0, resolution=4), -np.inf, 0, np.inf, np.inf),
    )

    resolutions = {"core": {"resolution": 0.1, "distance": 1}}

    mesh = from_meshio(mesh_from_OrderedDict(polygons, resolutions, default_resolution_max=0.6))

    basis0 = Basis(mesh, ElementTriP0())
    epsilon = basis0.zeros(dtype=complex)
    for subdomain, n in {"core": n_SiN, "box": n_SiO2, "clad": n_air}.items():
        epsilon[basis0.get_dofs(elements=subdomain)] = n**2

    modes = compute_modes(basis0, epsilon, wavelength=wavelength, num_modes=num_modes)
    all_neffs[i] = np.real([mode.n_eff for mode in modes])
    all_te_fracs[i, :] = [mode.te_fraction for mode in modes]

In [ ]:
all_neffs = np.real(all_neffs)
plt.xlabel("Width of waveguide / µm")
plt.ylabel("Effective refractive index")
plt.fill_between(widths, n_SiO2, alpha=0.5, color="gray")
plt.ylim(1.36, np.max(all_neffs) + 0.1 * (np.max(all_neffs) - n_SiO2))
for lams, te_fracs in zip(all_neffs.T, all_te_fracs.T):
    plt.plot(widths, lams)
    plt.scatter(widths, lams, c=te_fracs, cmap="cool")
plt.colorbar().set_label("TE fraction")
plt.show()

In [ ]:
cutoff_widths = np.full(all_neffs.shape[1], np.nan)

for m in range(all_neffs.shape[1]):        # loop over modes
    idx = np.where(all_neffs[:, m] > n_SiO2)[0]
    if idx.size > 0:
        cutoff_widths[m] = widths[idx[0]]  # first width where mode is guided

cutoff_widths

In [ ]:
print(f"The waveguide is single-mode for widths between {cutoff_widths[0]:.2f} µm and {cutoff_widths[1]:.2f} µm.")